# 05 — Modelagem e Avaliação

**Objetivo:** treinar uma regressão logística com fórmula (`statsmodels.formula.api`), avaliar significância, aplicar stepwise backward e comparar o modelo completo com o reduzido.

**Entrada:** `data/02_gold/*_selecionado.parquet` e `variaveis_selecionadas.json`  
**Saída:** modelos, histórico do stepwise, comparação, métricas, scores e lista final de variáveis.

In [ ]:
import json
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", None)

## 1. Carga das bases selecionadas

In [ ]:
treino = pd.read_parquet("../data/02_gold/treino_selecionado.parquet")
validacao = pd.read_parquet("../data/02_gold/validacao_selecionado.parquet")
teste = pd.read_parquet("../data/02_gold/teste_selecionado.parquet")

with open("../data/02_gold/variaveis_selecionadas.json") as file:
    variaveis = json.load(file)

target = "default"

print("Variáveis disponíveis:", len(variaveis))
print(variaveis)

## 2. Modelo completo com fórmula

In [ ]:
formula_completa = f"{target} ~ " + " + ".join(variaveis)
modelo_completo = smf.logit(formula_completa, data=treino).fit(disp=0)

modelo_completo.summary()

In [ ]:
tabela_completa = pd.DataFrame({
    "variavel": modelo_completo.params.index,
    "coef": modelo_completo.params.values,
    "p_value": modelo_completo.pvalues.values,
    "std_err": modelo_completo.bse.values,
    "z": modelo_completo.tvalues.values,
}).sort_values("p_value", ascending=False)

tabela_completa["significativo_5pct"] = tabela_completa["p_value"] < 0.05
tabela_completa

## 3. Stepwise backward por p-value

A cada iteração, remove a variável com maior p-value enquanto ela estiver acima de `0.05`. O processo usa **somente o treino**.

In [ ]:
def stepwise_logit(data, target, variaveis, limite_p=0.05):
    selecionadas = variaveis.copy()
    historico = []

    while True:
        formula = f"{target} ~ " + " + ".join(selecionadas)
        modelo = smf.logit(formula, data=data).fit(disp=0)
        pvalues = modelo.pvalues.drop("Intercept")
        maior_p = pvalues.max()

        if maior_p <= limite_p:
            break

        removida = pvalues.idxmax()
        historico.append({
            "variavel_removida": removida,
            "p_value": maior_p,
            "aic": modelo.aic,
            "bic": modelo.bic,
        })
        selecionadas.remove(removida)

    return selecionadas, modelo, pd.DataFrame(historico)

In [ ]:
variaveis_reduzidas, modelo_reduzido, historico_stepwise = stepwise_logit(
    treino,
    target,
    variaveis,
    limite_p=0.05,
)

print("Variáveis removidas:", historico_stepwise["variavel_removida"].tolist())
print("Variáveis mantidas:", variaveis_reduzidas)
historico_stepwise

## 4. Modelo reduzido

In [ ]:
formula_reduzida = f"{target} ~ " + " + ".join(variaveis_reduzidas)
modelo_reduzido = smf.logit(formula_reduzida, data=treino).fit(disp=0)

modelo_reduzido.summary()

In [ ]:
tabela_reduzida = pd.DataFrame({
    "variavel": modelo_reduzido.params.index,
    "coef": modelo_reduzido.params.values,
    "p_value": modelo_reduzido.pvalues.values,
    "std_err": modelo_reduzido.bse.values,
    "z": modelo_reduzido.tvalues.values,
}).sort_values("p_value", ascending=False)

tabela_reduzida["significativo_5pct"] = tabela_reduzida["p_value"] < 0.05
tabela_reduzida

In [ ]:
X_reduzido = treino[variaveis_reduzidas]
X_reduzido_centrado = X_reduzido - X_reduzido.mean()

vif_reduzido = pd.Series(
    [
        variance_inflation_factor(X_reduzido_centrado.values, i)
        for i in range(X_reduzido.shape[1])
    ],
    index=variaveis_reduzidas,
).sort_values(ascending=False)

vif_reduzido

## 5. Comparação entre modelo completo e reduzido

In [ ]:
def avaliar_modelo(modelo, base, nome):
    prob = modelo.predict(base)
    y = base[target]

    auc = roc_auc_score(y, prob)
    fpr, tpr, _ = roc_curve(y, prob)
    ks = max(tpr - fpr)
    gini = 2 * auc - 1

    return {
        "base": nome,
        "n": len(base),
        "bad_rate": y.mean(),
        "auc": auc,
        "ks": ks,
        "gini": gini,
    }, prob

In [ ]:
comparacao = []

for nome_modelo, modelo, variaveis_modelo in [
    ("completo", modelo_completo, variaveis),
    ("reduzido", modelo_reduzido, variaveis_reduzidas),
]:
    for nome_base, base in [("treino", treino), ("validacao", validacao)]:
        resumo, _ = avaliar_modelo(modelo, base, nome_base)
        resumo["modelo"] = nome_modelo
        resumo["n_variaveis"] = len(variaveis_modelo)
        resumo["aic"] = modelo.aic
        resumo["bic"] = modelo.bic
        comparacao.append(resumo)

comparacao_modelos = pd.DataFrame(comparacao)
comparacao_modelos = comparacao_modelos[[
    "modelo", "n_variaveis", "base", "n", "bad_rate",
    "auc", "ks", "gini", "aic", "bic",
]]

comparacao_modelos.round(4)

## 6. Escolha do modelo final

O modelo reduzido é escolhido por parcimônia. A avaliação abaixo mostra que a performance na validação é praticamente equivalente.

In [ ]:
modelo_final = modelo_reduzido
variaveis_finais = variaveis_reduzidas
formula_final = formula_reduzida

print("Modelo final: reduzido")
print("Variáveis finais:", len(variaveis_finais))
print(variaveis_finais)

## 7. Avaliação final no treino, validação e teste

In [ ]:
metricas = []
resultados = {}

for nome, base in [("treino", treino), ("validacao", validacao), ("teste", teste)]:
    resumo, prob = avaliar_modelo(modelo_final, base, nome)
    metricas.append(resumo)
    resultados[nome] = prob

metricas = pd.DataFrame(metricas)
metricas.round(4)

## 8. Curva ROC

In [ ]:
plt.figure(figsize=(7, 5))

for nome, base in [("treino", treino), ("validacao", validacao), ("teste", teste)]:
    fpr, tpr, _ = roc_curve(base[target], resultados[nome])
    auc = roc_auc_score(base[target], resultados[nome])
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("Falso positivo")
plt.ylabel("Verdadeiro positivo")
plt.title("Curva ROC — modelo final")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Ponto de corte na validação

In [ ]:
fpr_val, tpr_val, thresholds_val = roc_curve(validacao[target], resultados["validacao"])
indice_corte = np.argmax(tpr_val - fpr_val)
corte = thresholds_val[indice_corte]

print("Corte escolhido na validação:", round(corte, 4))

predito_teste = (resultados["teste"] >= corte).astype(int)
matriz_teste = confusion_matrix(teste[target], predito_teste)

pd.DataFrame(
    matriz_teste,
    index=["Real 0", "Real 1"],
    columns=["Predito 0", "Predito 1"],
)

## 10. Gravação dos resultados

In [ ]:
comparacao_modelos.to_csv("../data/02_gold/comparacao_modelos.csv", index=False)
historico_stepwise.to_csv("../data/02_gold/stepwise_historico.csv", index=False)
metricas.to_csv("../data/02_gold/metricas_modelo.csv", index=False)

with open("../data/02_gold/variaveis_stepwise.json", "w") as file:
    json.dump(variaveis_finais, file, indent=2)

with open("../data/02_gold/modelo_logistico_completo.pkl", "wb") as file:
    pickle.dump(modelo_completo, file, protocol=pickle.HIGHEST_PROTOCOL)

with open("../data/02_gold/modelo_logistico_reduzido.pkl", "wb") as file:
    pickle.dump(modelo_reduzido, file, protocol=pickle.HIGHEST_PROTOCOL)

with open("../data/02_gold/modelo_logistico.pkl", "wb") as file:
    pickle.dump(modelo_final, file, protocol=pickle.HIGHEST_PROTOCOL)

scores_teste = teste[["ID", "SAFRA", target]].copy()
scores_teste["prob_default"] = resultados["teste"]
scores_teste["predito"] = (scores_teste["prob_default"] >= corte).astype(int)
scores_teste.to_parquet("../data/02_gold/scores_teste.parquet", index=False)

print("Arquivos gravados em data/02_gold")

In [ ]:
metricas.round(4)